In [4]:
import requests

BASE = "https://api.openalex.org"
MY_INST = "3Ai4405392" # BOUN

def api(endpoint, params):
    return requests.get(f"{BASE}/{endpoint}", params=params).json()


In [5]:
# Get all co-authoring institutions in one call
collab_data = api("works", {
    "filter": f"authorships.institutions.id:{MY_INST},publication_year:2020-",
    "group_by": "authorships.institutions.id",
    "per_page": 100,
})["group_by"]

collabs = {g["key"].split("/")[-1]: g["count"] for g in collab_data}
collabs

{'I4405392': 6398,
 'I1294671590': 366,
 'I67581229': 352,
 'I67311998': 333,
 'I27837315': 330,
 'I2802243403': 326,
 'I16391192': 322,
 'I48912391': 318,
 'I79619799': 316,
 'I78577930': 315,
 'I212119943': 312,
 'I86519309': 311,
 'I174458059': 310,
 'I5023651': 310,
 'I91136226': 310,
 'I102502594': 309,
 'I277688954': 309,
 'I184558857': 308,
 'I114457229': 307,
 'I162608824': 306,
 'I187531555': 306,
 'I201448701': 306,
 'I21491767': 306,
 'I8692664': 306,
 'I185261750': 305,
 'I686019': 305,
 'I74656192': 305,
 'I151201029': 303,
 'I122140584': 302,
 'I138164181': 302,
 'I4210148700': 301,
 'I185103710': 300,
 'I4210108560': 300,
 'I4210136779': 300,
 'I4210142660': 300,
 'I4210160382': 300,
 'I2800204708': 299,
 'I4210113126': 299,
 'I874386039': 299,
 'I1286704778': 298,
 'I4210092312': 298,
 'I16097986': 296,
 'I4403928244': 286,
 'I70900168': 285,
 'I4210151851': 284,
 'I174306211': 281,
 'I138728355': 277,
 'I63634437': 273,
 'I1351752': 269,
 'I197323543': 268,
 'I20077721

In [6]:
# Get top 5 topics
topics = api("works", {
    "filter": f"authorships.institutions.id:{MY_INST}",
    "group_by": "primary_topic.id",
    "per_page": 5,
})["group_by"]

topics

[{'key': 'https://openalex.org/T10048',
  'key_display_name': 'Particle physics theoretical and experimental studies',
  'count': 999},
 {'key': 'https://openalex.org/T10110',
  'key_display_name': 'earthquake and tectonic studies',
  'count': 402},
 {'key': 'https://openalex.org/T11783',
  'key_display_name': "Turkey's Politics and Society",
  'count': 270},
 {'key': 'https://openalex.org/T10323',
  'key_display_name': 'Analog and Mixed-Signal Circuit Design',
  'count': 189},
 {'key': 'https://openalex.org/T10160',
  'key_display_name': 'Seismic Performance and Analysis',
  'count': 168}]

In [7]:
for topic in topics:
    tid = topic["key"].split("/")[-1]
    print(f"\n=== {topic['key_display_name']} ({topic['count']} works) ===")

    # Top institutions in this topic
    top_insts = api("works", {
        "filter": f"primary_topic.id:{tid}",
        "group_by": "authorships.institutions.id",
        "per_page": 15,
    })["group_by"]

    # Recent vs. historical for rising star detection
    recent = {g["key"].split("/")[-1] for g in api("works", {
        "filter": f"primary_topic.id:{tid},publication_year:2023-2025",
        "group_by": "authorships.institutions.id",
        "per_page": 15,
    })["group_by"]}

    historical = {g["key"].split("/")[-1] for g in api("works", {
        "filter": f"primary_topic.id:{tid},publication_year:2018-2020",
        "group_by": "authorships.institutions.id",
        "per_page": 15,
    })["group_by"]}

    rising = recent - historical

    for inst in top_insts:
        iid = inst["key"].split("/")[-1]
        if iid == MY_INST:
            continue

        coauthored = collabs.get(iid, 0)
        tag = " ** RISING **" if iid in rising else ""
        print(f"  {inst['key_display_name']}: "
              f"{inst['count']} topic works, {coauthored} co-authored{tag}")


=== Particle physics theoretical and experimental studies (999 works) ===
  European Organization for Nuclear Research: 14068 topic works, 333 co-authored
  Centre National de la Recherche Scientifique: 7837 topic works, 366 co-authored
  Fermi National Accelerator Laboratory: 6913 topic works, 0 co-authored
  Joint Institute for Nuclear Research: 5245 topic works, 163 co-authored
  Institut National de Physique Nucléaire et de Physique des Particules: 4977 topic works, 261 co-authored
  Deutsches Elektronen-Synchrotron DESY: 4494 topic works, 186 co-authored
  Istituto Nazionale di Fisica Nucleare, Laboratori Nazionali di Frascati: 4486 topic works, 300 co-authored
  Lawrence Berkeley National Laboratory: 4233 topic works, 163 co-authored
  Brookhaven National Laboratory: 4217 topic works, 263 co-authored
  University of Michigan: 3815 topic works, 330 co-authored
  High Energy Accelerator Research Organization: 3807 topic works, 277 co-authored
  The University of Tokyo: 3670 topic 